In [1]:
# Import packages
import json
from huggingface_hub import login
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, pipeline
import transformers
import random
import torch
import time
import re
from tqdm import tqdm
import pandas as pd

# Set GPUs
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

print("Available GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Set seeds
random.seed(0)
torch.manual_seed(0)

/home/sswee/miniconda3/envs/shdb-af-analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Available GPUs: 4
GPU 0: NVIDIA GeForce RTX 2080 Ti
GPU 1: NVIDIA GeForce RTX 2080 Ti
GPU 2: NVIDIA GeForce RTX 2080 Ti
GPU 3: NVIDIA GeForce RTX 2080 Ti


In [3]:
# Log into Huggingface
with open("../../huggingface_token.txt", "r") as file:
    access_token = file.read().strip()
login(access_token)

# Load Huggingface Model
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, token=access_token, low_cpu_mem_usage=True,
                    torch_dtype=torch.float16, device_map='auto')

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /home/sswee/.cache/huggingface/token
Login successful


Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [05:08<00:00, 77.02s/it]


In [75]:
def get_message(note):
    system = """
You are a cardiologist.

The patient is from a heart failure clinic population.
In this population, reduced LVEF, elevated BNP, and NYHA II–III symptoms are common.
Do NOT treat these findings as automatically implying imminent death.
Consider realistic multi-year outcomes in treated heart failure patients.

Your task is to assess risk over the next few years.

There are TWO independent outcomes:

1) sudden cardiac death
2) pump failure death

You MUST:

1) Assess SCD risk.
2) Assess PFD risk.
3) Use ONLY the provided patient data.
4) Do NOT assume missing findings.
5) Output EXACTLY in the format below and nothing else.

Output format (strict):

SCD:
Risk: <low | moderate | high>
Explanation: <detailed reasoning>

PFD:
Risk: <low | moderate | high>
Explanation: <detailed reasoning>

Do not output anything else.
"""

    prompt = f"Patient data:\n{note}"

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]

FULL_PATTERN = re.compile(
    r"SCD:\s*"
    r"Risk:\s*(low|moderate|high)\s*"
    r"Explanation:\s*(.*?)\s*"
    r"PFD:\s*"
    r"Risk:\s*(low|moderate|high)\s*"
    r"Explanation:\s*(.*)",
    flags=re.DOTALL | re.IGNORECASE
)

def extract_assistant_response(text: str):
    m = FULL_PATTERN.search(text)
    if not m:
        return None

    return {
        "scd_risk": m.group(1).lower(),
        "scd_reasoning": m.group(2).strip(),
        "pfd_risk": m.group(3).lower(),
        "pfd_reasoning": m.group(4).strip(),
    }


In [77]:
# Load in csv file with prompts
df = pd.read_csv("../../music/subject-info-cleaned-with-prompts.csv")
# print(df['Prompts'][0])

In [81]:
# Test message
message = get_message(df['Prompts'][1])
print(message)

[{'role': 'system', 'content': '\nYou are a cardiologist.\n\nThe patient is from a heart failure clinic population.\nIn this population, reduced LVEF, elevated BNP, and NYHA II–III symptoms are common.\nDo NOT treat these findings as automatically implying imminent death.\nConsider realistic multi-year outcomes in treated heart failure patients.\n\nYour task is to assess risk over the next few years.\n\nThere are TWO independent outcomes:\n\n1) sudden cardiac death\n2) pump failure death\n\nYou MUST:\n\n1) Assess SCD risk.\n2) Assess PFD risk.\n3) Use ONLY the provided patient data.\n4) Do NOT assume missing findings.\n5) Output EXACTLY in the format below and nothing else.\n\nOutput format (strict):\n\nSCD:\nRisk: <low | moderate | high>\nExplanation: <detailed reasoning>\n\nPFD:\nRisk: <low | moderate | high>\nExplanation: <detailed reasoning>\n\nDo not output anything else.\n'}, {'role': 'user', 'content': 'Patient data:\nAge: 58.0\nGender: Male \nWeight: 74 kg\nHeight: 160 cm\nNYHA

In [82]:
# # Put message into LLM
# input_text = tokenizer.apply_chat_template(message, tokenize = False, add_generation_prompt = True)
# inputs = tokenizer(input_text, return_tensors = "pt").to(model.device)
# output = model.generate(**inputs, max_new_tokens = 1000)

# Put message into LLM
input_text = tokenizer.apply_chat_template(
    message,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=False,             
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(output[0], skip_special_tokens=True)


In [83]:
# Get result
result = tokenizer.decode(output[0], skip_special_tokens = True)
result = result.replace("**", "")
result = extract_assistant_response(result)
print(result)

{'scd_risk': 'moderate', 'scd_reasoning': 'The patient has a history of ischemic dilated cardiomyopathy and myocardial infarction, which increases the risk of sudden cardiac death. However, the patient is on beta blockers and an angiotensin II receptor blocker, which are known to reduce the risk of sudden cardiac death. The LVEF is 35%, which is low but not extremely low. The absence of sustained ventricular tachycardia and other high-risk ECG findings also reduces the risk. However, the presence of monomorphic ventricular extrasystoles may indicate underlying electrical instability, increasing the risk of sudden cardiac death.', 'pfd_risk': 'high', 'pfd_reasoning': "The patient has a low LVEF of 35%, which is a strong indicator of pump failure risk. The elevated Pro-BNP level of 570 ng/L also supports this risk. The patient's NYHA class is II, indicating some symptoms but not severe, which may suggest that the patient is not yet in an advanced stage of heart failure. However, the pati